# Leapfrog algorithm theory

We are using this in a NVE ensemble (not NVT). For this ensemble the code given by prof. Keller uses different steps to construct the Verlet algorithm. The snippet of the code is given in the cell below:

If we take a look at the code snippets of the steps we can see how the equations fit into the leapfrog integrator.
1. __A-step__: in this step the positions are updated for a full ($\Delta t$) or half time step ($\frac{\Delta t}{2}$), using velocities at current ($t$) time. This code should be used to calculate the position update using velocities that are half a step ahead ($v(t+\frac{\Delta}{2})$).
2. __B-step__: in this step the velocities are updated for a for a full ($\Delta t$) or half time step ($\frac{\Delta t}{2}$) using the force that was calculated at time $t$. 

The convenient thing about both A and B steps that they already have an option to switch on a half a time step using `half_step == True`. Now we just have to construct the code together so that the leapfrog algorithm is used properly.

In [ ]:
def A_step(ps: ParticleSystem, sim: SimulationParameters, half_step=False):
    """
    Performs the A-step (position update) of an MD integration scheme.

    This step updates particle positions using the current velocities:
    r(t + Δt) = r(t) + v(t) * Δt

    Parameters:
        - ps (ParticleSystem): The particle system containing positions and velocities.
        - sim (SimulationParameters): Simulation settings, including the time step.
        - half_step (bool): If True, performs a half step (Δt / 2) instead of a full step.

    Returns:
        None. Updates ps.position in-place.
    """
    # set time step, depending on whether a half- or full step is performed
    if half_step == True:
        dt = 0.5 * sim.dt
    else:
        dt = sim.dt
        
    ps.position = ps.position + ps.velocity * dt
    
    return None    

def B_step(ps: ParticleSystem, sim: SimulationParameters, half_step=False):
    """
    Performs the B-step (velocity update) of an MD integration scheme.

    This step updates particle velocities using the current forces:
    v(t + Δt) = v(t) + 1/m * Δt * F(t) 
 
    Parameters:
        - ps (ParticleSystem): The particle system containing positions and velocities.
        - sim (SimulationParameters): Simulation settings, including the time step.
        - half_step (bool): If True, performs a half step (Δt / 2) instead of a full step.

    Returns:
        None. Updates ps.velocity in-place.
    """
    # set time step, depending on whether a half- or full step is performed
    if half_step == True:
        dt = 0.5 * sim.dt
    else:
        dt = sim.dt
        
    # (1/ps.mass)[:, np.newaxis] = explicit reshaping to avoid
    # broadcasting issues when multiplying (N,) with (N,3) elementwise
    # now it is explicit: (N,1) * (N,3)
    ps.velocity = ps.velocity + (1/ps.mass)[:, np.newaxis]* dt * ps.force 
    
    return None   

Here is the snippet of the BAB Verlet integrator code:

In [ ]:
def simulate_NVE_step(ps: ParticleSystem, sim: SimulationParameters):
    """
    Performs a single time step of molecular dynamics in the NVE ensemble
    using the velocity Verlet integrator in BAB form (half-step B, full-step A, half-step B).

    The steps are:
    1. Half-step velocity update (B-step)
    2. Full-step position update (A-step)
    3. Force recalculation based on new positions
    4. Second half-step velocity update (B-step)
    5. Apply periodic boundary conditions

    This corresponds to a time-symmetric, second-order accurate integrator for Newtonian dynamics.

    Parameters:
        - ps (ParticleSystem): The particle system containing positions, velocities, and forces.
        - sim (SimulationParameters): Simulation parameters including time step.

    Returns:
        None. Updates ps.position, ps.velocity, and ps.force in-place.
    """
    B_step(ps, sim, half_step=True)   # update velocity by a half-step
    A_step(ps, sim, half_step=False)  # update position by a full time step
    calculate_force(ps, sim)          # udpate force  
    B_step(ps, sim, half_step=True)   # update velocity by a second half-step

    apply_periodic_boundary(ps, sim)
        
    return None  

## Leapfrog algorithm outline:
The idea of the leapfrog algorithm is to evaluate the positions and velocities at different points in time, they are offset by half a timestep and thus they leapfrog over each other. Initially we need to offset the velocity by half a time-step (just in the initialization part). See below for the explanation of the algorithm.

These are the equations for the leapfrog algorithm:
$$
\mathbf{v}\left(t + \frac{1}{2}\Delta t\right)
=
\mathbf{v}\left(t - \frac{1}{2}\Delta t\right)
+
\frac{\mathbf{F}(t)}{m}\,\Delta t
$$

$$
\mathbf{r}(t + \Delta t)
=
\mathbf{r}(t)
+
\mathbf{v}\left(t + \frac{1}{2}\Delta t\right)\,\Delta t
$$

Points 1 and 2 are done before the loop.
1. Calculate the force according to the initial positions F(0). This is already done in the run script in the LJ_gas_run_MD.py (line 120): 
```python
# calculate force according to initial positions
calculate_force(ps, sim)
```
2. Do a half step velocity update (B-step) using `half_step == True`. This updates $v(0)$-->v($0+\frac{1}{2}\Delta t$)

__In the actual MD loop__:

3. Do a full step position update (A-step, `half_step == False`) using the half-step velocity update v($0+\frac{1}{2}\Delta t$). This moves positions from $r(0)$ --> $r(0+\Delta t)$
4. Calculate the force at time $F(0+\Delta t$) from the new positions in step 3.
5. Do a full B step: since we have the force we can move the velocities to a full time-step ahead: $v(0+\frac{1}{2}\Delta t)$-->$v(0+\frac{3}{2}\Delta t)$
6. Because we have moved the particles and updated the velocities we need to apply periodic boundary conditions.
So we see that we have positions at integer times but the velocities at half-integer times, so they leapfrog cleanly over each other.
7. To track the kinetic energy at the integer time (at which we store positions) we need to average the velocities before and after the B step. That way energy calculations are synchronized (hasn't been done yet).

__Averaging the velocities to get accurate energy results__

In the leapfrog integrator the velocities are calculated at half-time steps and not integer time step values, so if we just leave it like this the kinetic energy calculation would be shifted since they wouldn't be calculated at the same point in time as the positions. To correct this we need to (linearly) interpolate between the velocity values for each time step. This can be done using this equation:
$$
\mathbf{v}(t + \Delta t)
\approx
\frac{
\mathbf{v}\left(t + \frac{1}{2}\Delta t\right)
+
\mathbf{v}\left(t + \frac{3}{2}\Delta t\right)
}{2}
$$ 

In the code this is implemnted in such a way:
1. Before the leapfrog step is calculated the program copies the velocity (which is at half-integer step, $t+\frac{1}{2} \Delta t$). This is saved as `v_before`.
2. Simulate a leapfrog step. In the step we obtain the velocity at $t+\frac{3}{2} \Delta t$. This is saved as `v_after`.
3. Now we calculate the average value between the two (as per equation above). So we get a velocity estimate at integer time step, `v_sync`.
4. We for the next step we also the save the velocity that will be used in the next iteration, `v_actual`.
5. Crucially, the program temporarily swaps the `v_actual` with `v_sync`. This is executed with the line `ps.velocity = v_sync`.
6. The code calls the functions `potential_energy`, `kinetic_energy`, `instantaneous_temperature` and `ideal_gas_pressure` to calculate the energies in the current integration step, using `v_sync`.
7. The synced velocity (at integer time step, `v_sync`) is again swapped with the actual velocity that it's used for integration, `v_actual`.